# Modular Analogy Generation Pipeline

This notebook provides an interface for testing different module configurations and generating scientific analogies.

## Features
- **Modular Design**: Each module can be swapped with different implementations
- **Configurable Ordering**: Reorder modules as needed
- **Multiple Input Formats**: Test with different input configurations
- **Comprehensive Evaluation**: Compare against baselines and SCAR golden standard
- **Experiment Tracking**: Save results and configurations


In [ ]:
# Setup and Imports
import sys
from pathlib import Path
import pandas as pd
import json

# Add current directory to path
current_dir = Path.cwd()
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

# Import pipeline components
from pipeline_config import PipelineConfig, InputFormat, get_default_config, get_full_config
from pipeline_runner import PipelineRunner

print("✅ Imports successful")


## Configuration

Choose your pipeline configuration:
- **Default Config**: Source Finder → Property Matcher → Explanation Generator
- **Full Config**: All modules in sequence
- **Custom Config**: Build your own configuration


In [ ]:
# Choose configuration type
CONFIG_TYPE = "default"  # Options: "default", "full", "custom"

if CONFIG_TYPE == "default":
    config = get_default_config()
    print("Using default configuration")
elif CONFIG_TYPE == "full":
    config = get_full_config()
    print("Using full configuration")
else:
    # Custom configuration
    config = PipelineConfig()
    
    # Add modules in desired order
    config.add_module(
        "source_finder",
        "EmbeddingSourceFinder",
        corpus_path="../../data/SCAR_cleaned_manually.csv",
        embedding_mode="name_background",
        top_k=10
    )
    config.add_module(
        "property_matcher",
        "DSPyPropertyMatcher",
        model_name="gpt-4o-mini",
        use_description=True
    )
    config.add_module(
        "explanation_generator",
        "DSPyExplanationGenerator",
        model_name="gpt-4o-mini",
        use_description=True,
        use_paired_properties=True
    )
    print("Using custom configuration")

# Configure input format
config.input_format = InputFormat.TARGET_PROPERTIES_DESCRIPTION  # Options: TARGET_ONLY, TARGET_PROPERTIES, TARGET_DESCRIPTION, TARGET_PROPERTIES_DESCRIPTION

# Configure evaluation
config.run_baselines = True
config.run_scar_evaluation = True
config.llm_judge_threshold = 0.7

# Experiment settings
config.experiment_name = "test_experiment"
config.save_results = True
config.results_dir = "./results"

# Display configuration
print("\n📋 Configuration:")
print(f"  Input Format: {config.input_format.value}")
print(f"  Modules ({len(config.get_enabled_modules())}):")
for i, m in enumerate(config.get_enabled_modules(), 1):
    print(f"    {i}. {m.module_type} ({m.implementation})")
print(f"  Run Baselines: {config.run_baselines}")
print(f"  Run SCAR Evaluation: {config.run_scar_evaluation}")
print(f"  LLM Judge Threshold: {config.llm_judge_threshold}")


## Initialize Pipeline

Create the pipeline runner with the selected configuration.


In [ ]:
# Initialize pipeline runner
runner = PipelineRunner(config)

print("✅ Pipeline initialized")
print(f"   Modules loaded: {len(runner.modules)}")
for module in runner.modules:
    print(f"   - {module.name}")


## Run Single Example

Test the pipeline on a single target concept.


In [ ]:
# Example input
target_name = "biological clock"
target_description = "The biological clock is a fundamental aspect of human physiology that governs the body's internal rhythms and processes."
target_properties = ["changes", "state", "adjust"]

# Run pipeline
result = runner.run(
    target_name=target_name,
    target_description=target_description,
    target_properties=target_properties,
    return_intermediate=False
)

# Display results
print("🎯 Results:")
print(f"\n📌 Target: {result['target_name']}")
print(f"📌 Analogy Type: {result.get('analogy_type', 'N/A')}")

if result.get('selected_source'):
    source = result['selected_source']
    print(f"\n🔍 Selected Source:")
    print(f"   Name: {source.get('name', 'N/A')}")
    print(f"   Domain: {source.get('domain', 'N/A')}")
    print(f"   Score: {source.get('score', 'N/A')}")

if result.get('property_mappings'):
    print(f"\n🔗 Property Mappings ({len(result['property_mappings'])}):")
    for mapping in result['property_mappings']:
        print(f"   {mapping[0]} → {mapping[1]}")

if result.get('explanation'):
    print(f"\n📝 Explanation:")
    if isinstance(result['explanation'], list):
        for i, exp in enumerate(result['explanation'], 1):
            print(f"   {i}. {exp}")
    else:
        print(f"   {result['explanation']}")

if result.get('evaluation_scores'):
    print(f"\n⭐ Evaluation Scores:")
    for key, value in result['evaluation_scores'].items():
        print(f"   {key}: {value:.3f}")

if result.get('scar_evaluation'):
    scar = result['scar_evaluation']
    print(f"\n🏆 SCAR Evaluation:")
    print(f"   Source Match: {scar.get('source_match', False)}")
    print(f"   Mapping Match: {scar.get('mapping_match', False)}")
    print(f"   Overall Match: {scar.get('overall_match', False)}")
    if scar.get('llm_judge_score') is not None:
        print(f"   LLM Judge Score: {scar.get('llm_judge_score', 0):.3f}")
        print(f"   LLM Judge Passed: {scar.get('llm_judge_passed', False)}")


## Run Batch Evaluation

Run the pipeline on multiple examples from SCAR dataset.


In [ ]:
# Load SCAR dataset
scar_path = "../../data/SCAR_cleaned_manually.csv"
df_scar = pd.read_csv(scar_path)

# Parse mappings to extract properties
import ast
df_scar['mappings_list'] = df_scar['mappings_parsed'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) and x else []
)

# Extract target properties from mappings
df_scar['target_properties'] = df_scar['mappings_list'].apply(
    lambda x: [pair[0] for pair in x] if x else []
)

print(f"✅ Loaded {len(df_scar)} SCAR records")
print(f"\nSample record:")
print(f"  Target: {df_scar.iloc[0]['system_a']}")
print(f"  Golden Source: {df_scar.iloc[0]['system_b']}")
print(f"  Properties: {df_scar.iloc[0]['target_properties']}")


In [ ]:
# Prepare inputs for batch processing
# Use a subset for testing (change NUM_SAMPLES to test more)
NUM_SAMPLES = 5  # Set to None to process all

if NUM_SAMPLES:
    df_sample = df_scar.head(NUM_SAMPLES)
else:
    df_sample = df_scar

inputs = []
for _, row in df_sample.iterrows():
    inputs.append({
        'target_name': row['system_a'],
        'target_description': row['system_a_background'],
        'target_properties': row['target_properties']
    })

print(f"📊 Processing {len(inputs)} examples...")


In [ ]:
# Run batch processing
results = runner.run_batch(inputs, save_results=True)

print(f"\n✅ Processed {len(results)} examples")
print(f"📁 Results saved to: {config.results_dir}")


## Analyze Results

Analyze the batch results and compare against golden standard.


In [ ]:
# Analyze results
if results:
    # Calculate metrics
    total = len(results)
    source_matches = sum(1 for r in results if r.get('scar_evaluation', {}).get('source_match', False))
    mapping_matches = sum(1 for r in results if r.get('scar_evaluation', {}).get('mapping_match', False))
    overall_matches = sum(1 for r in results if r.get('scar_evaluation', {}).get('overall_match', False))
    
    # LLM judge scores
    llm_scores = [r.get('scar_evaluation', {}).get('llm_judge_score') 
                  for r in results 
                  if r.get('scar_evaluation', {}).get('llm_judge_score') is not None]
    avg_llm_score = sum(llm_scores) / len(llm_scores) if llm_scores else 0.0
    
    # Evaluation scores
    eval_scores = [r.get('evaluation_scores', {}).get('overall_score', 0.0) 
                   for r in results 
                   if r.get('evaluation_scores', {}).get('overall_score') is not None]
    avg_eval_score = sum(eval_scores) / len(eval_scores) if eval_scores else 0.0
    
    print("📊 Results Summary:")
    print(f"   Total Examples: {total}")
    print(f"   Source Matches: {source_matches} ({source_matches/total*100:.1f}%)")
    print(f"   Mapping Matches: {mapping_matches} ({mapping_matches/total*100:.1f}%)")
    print(f"   Overall Matches: {overall_matches} ({overall_matches/total*100:.1f}%)")
    print(f"   Avg LLM Judge Score: {avg_llm_score:.3f}")
    print(f"   Avg Evaluation Score: {avg_eval_score:.3f}")
    
    # Show examples
    print("\n📋 Example Results:")
    for i, result in enumerate(results[:3], 1):
        print(f"\n   Example {i}:")
        print(f"   Target: {result.get('target_name', 'N/A')}")
        if result.get('selected_source'):
            print(f"   Generated Source: {result['selected_source'].get('name', 'N/A')}")
        scar = result.get('scar_evaluation', {})
        if scar:
            print(f"   Golden Source: {scar.get('golden_source', 'N/A')}")
            print(f"   Source Match: {scar.get('source_match', False)}")
            print(f"   LLM Judge Score: {scar.get('llm_judge_score', 'N/A')}")


In [ ]:
# Compare with baselines
if results and results[0].get('baseline_results'):
    print("📊 Baseline Comparison:")
    
    # Count baseline matches
    baseline_embedding_matches = 0
    baseline_llm_matches = 0
    
    for result in results:
        target_name = result.get('target_name', '')
        golden_source = result.get('scar_evaluation', {}).get('golden_source', '')
        
        if not golden_source:
            continue
        
        # Check embedding baseline
        if 'embedding' in result.get('baseline_results', {}):
            embedding_sources = [s.get('name', '') for s in result['baseline_results']['embedding']]
            if any(golden_source.lower() in s.lower() or s.lower() in golden_source.lower() 
                   for s in embedding_sources):
                baseline_embedding_matches += 1
        
        # Check embedding+LLM baseline
        if 'embedding_llm' in result.get('baseline_results', {}):
            llm_sources = [s.get('name', '') for s in result['baseline_results']['embedding_llm']]
            if any(golden_source.lower() in s.lower() or s.lower() in golden_source.lower() 
                   for s in llm_sources):
                baseline_llm_matches += 1
    
    total = len([r for r in results if r.get('scar_evaluation', {}).get('golden_source')])
    
    print(f"   Embedding Baseline Matches: {baseline_embedding_matches}/{total} ({baseline_embedding_matches/total*100:.1f}%)")
    print(f"   Embedding+LLM Baseline Matches: {baseline_llm_matches}/{total} ({baseline_llm_matches/total*100:.1f}%)")
    print(f"   Pipeline Matches: {overall_matches}/{total} ({overall_matches/total*100:.1f}%)")
else:
    print("⚠️  Baseline results not available")
